# Experimento TCC — Correção Automatizada com e sem RAG

Este notebook executa o experimento completo do TCC via API do backend.

**Fluxo:**
1. Setup (registro, login, criação de turma e alunos)
2. Condição A — Com RAG (upload de PDF + publish)
3. Condição B — Sem RAG (publish sem PDF)
4. QA4 — Estabilidade (5 execuções da Condição A)
5. Coleta e exportação dos resultados

## 0. Configuração

In [17]:
import requests
import json
import time
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

BASE_URL = "http://localhost:8000"
PDF_PATH = Path("../notebooks/algoritmos-ufpr.pdf")
OUTPUT_DIR = Path("../notebooks/results")
OUTPUT_DIR.mkdir(exist_ok=True)

TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

assert PDF_PATH.exists(), f"PDF não encontrado em {PDF_PATH.resolve()}"

# Estado global do experimento
state = {}

def api(method, path, **kwargs):
    """Helper para chamadas à API com auth automático."""
    headers = kwargs.pop("headers", {})
    if "token" in state:
        headers["Authorization"] = f"Bearer {state['token']}"
    resp = getattr(requests, method)(f"{BASE_URL}{path}", headers=headers, **kwargs)
    if not resp.ok:
        print(f"❌ {method.upper()} {path} → {resp.status_code}")
        print(resp.text[:500])
    return resp

def save_csv(df, name):
    """Salva DataFrame como CSV em OUTPUT_DIR."""
    path = OUTPUT_DIR / f"{name}_{TIMESTAMP}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"  💾 {path}")
    return path

print("✅ Configuração carregada")
print(f"   Backend: {BASE_URL}")
print(f"   PDF:     {PDF_PATH.resolve()}")
print(f"   Output:  {OUTPUT_DIR.resolve()}")
print(f"   Run ID:  {TIMESTAMP}")

✅ Configuração carregada
   Backend: http://localhost:8000
   PDF:     C:\Users\Maycon Mendes\OneDrive\Área de Trabalho\IFF\TCC\ai-grading-system\notebooks\algoritmos-ufpr.pdf
   Output:  C:\Users\Maycon Mendes\OneDrive\Área de Trabalho\IFF\TCC\ai-grading-system\notebooks\results
   Run ID:  20260328_163833


## 1. Dados do Experimento

Questões, critérios e respostas

In [18]:
# ── Questões ──────────────────────────────────────────────────────────────────

QUESTIONS = [
    {
        "id": "Q1",
        "order": 1,
        "points": 10.0,
        "statement": (
            "Diferencie uma `Array` (vetor estático) de uma `Lista Encadeada Simples` "
            "quanto aos seguintes aspectos: alocação de memória, acesso a um elemento "
            "em uma posição arbitrária (índice `k`), e impacto da inserção/remoção de "
            "um elemento no meio da estrutura. Justifique suas respostas com exemplos "
            "de complexidade de tempo O(n)."
        ),
    },
    {
        "id": "Q2",
        "order": 2,
        "points": 10.0,
        "statement": (
            "Compare os algoritmos de ordenação `Merge Sort` e `Quick Sort` considerando "
            "os seguintes critérios: estabilidade, complexidade de tempo no pior caso e "
            "complexidade de espaço. Explique as implicações de cada característica para "
            "a escolha de um algoritmo em cenários específicos."
        ),
    },
    {
        "id": "Q3",
        "order": 3,
        "points": 10.0,
        "statement": (
            "Descreva as propriedades que definem uma `Árvore Binária de Busca (BST)` "
            "balanceada e uma não balanceada. Explique qual a principal vantagem de uma "
            "BST balanceada em relação a uma não balanceada, especificando a complexidade "
            "de tempo de busca, inserção e remoção em ambos os cenários (melhor e pior caso)."
        ),
    },
]

# ── Critérios de correção (sem UUID — serão buscados do banco em tempo de execução) ──
# Critérios selecionados para questões técnicas de Estruturas de Dados/Algoritmos.
# Pesos: Correção técnica 35%, Completude 25%, Precisão terminológica 20%,
#        Uso de exemplos 10%, Análise crítica 10%.
# max_points = weight * 10.0 (pontos totais da questão)

CRITERIAS = [
    {
        "name": "Correção técnica",
        "description": (
            "Avalia se os conceitos, complexidades e propriedades técnicas apresentados "
            "estão corretos e sem contradições ou erros factuais graves."
        ),
        "max_points": 3.5,   # 0.35 * 10
        "weight": 0.35,
    },
    {
        "name": "Completude da resposta",
        "description": (
            "Avalia se todos os pontos solicitados no enunciado foram abordados "
            "de forma adequada, sem omitir aspectos relevantes."
        ),
        "max_points": 2.5,   # 0.25 * 10
        "weight": 0.25,
    },
    {
        "name": "Precisão terminológica",
        "description": (
            "Avalia o uso correto de termos técnicos e definições da área, "
            "evitando ambiguidade, imprecisão ou uso coloquial de conceitos formais."
        ),
        "max_points": 2.0,   # 0.20 * 10
        "weight": 0.20,
    },
    {
        "name": "Uso de exemplos",
        "description": (
            "Avalia se a resposta fundamenta suas afirmações com exemplos concretos, "
            "notações de complexidade (Big-O) ou casos de uso que ilustrem o raciocínio."
        ),
        "max_points": 1.0,   # 0.10 * 10
        "weight": 0.10,
    },
    {
        "name": "Análise crítica e profundidade",
        "description": (
            "Avalia se o aluno vai além da definição superficial, discutindo implicações, "
            "trade-offs ou comparações entre alternativas de forma fundamentada."
        ),
        "max_points": 1.0,   # 0.10 * 10
        "weight": 0.10,
    },
]

# ── Alunos ────────────────────────────────────────────────────────────────────

STUDENTS = [
    {"name": "Aluno 1 (Excelente)",     "email": "aluno1.excelente@example.com", "profile": "Excelente"},
    {"name": "Aluno 2 (Intermediário)", "email": "aluno2.adequado@example.com",  "profile": "Intermediário"},
    {"name": "Aluno 3 (Fraco)",         "email": "aluno3.fraco@example.com",     "profile": "Fraca"},
    {"name": "Aluno 4 (Fora do Tema)",  "email": "aluno4.foratema@example.com",  "profile": "Fora tema"},
]

# Mapeamentos reutilizáveis
PROFILE_MAP = {s["name"]: s["profile"] for s in STUDENTS}
QUESTION_ORDER_TO_ID = {q["order"]: q["id"] for q in QUESTIONS}

# ── Respostas (12 = 3 questões × 4 alunos) ──────────────────────────────────
# Perfis:
#   Excelente     → resposta completa, terminologia precisa, Big-O corretos,
#                   análise de trade-offs e implicações práticas
#   Intermediário → conceitos corretos mas superficiais, sem detalhamento,
#                   sem exemplos concretos nem análise de implicações
#   Fraco         → erros conceituais graves (complexidades invertidas,
#                   propriedades erradas, terminologia incorreta)
#   Fora do tema  → resposta sobre assunto completamente diferente do enunciado

ANSWERS = {
    # ── Q1: Array vs Lista Encadeada ──────────────────────────────────────────
    (1, 0): (
        # Excelente
        "Arrays utilizam alocação contígua de memória: os elementos ocupam posições "
        "consecutivas, permitindo acesso direto via aritmética de ponteiros com "
        "complexidade O(1) para qualquer índice k. Listas Encadeadas Simples alocam "
        "nós dinamicamente em posições não contíguas; cada nó armazena o dado e um "
        "ponteiro para o próximo. O acesso ao índice k exige travessia sequencial a "
        "partir da cabeça, resultando em O(n) no pior caso. "
        "Na inserção/remoção no meio: em Arrays, todos os elementos após a posição "
        "precisam ser deslocados, custo O(n). Em Listas, uma vez localizado o nó "
        "predecessor (O(n) de busca), a operação em si é O(1) — apenas ponteiros são "
        "redirecionados. O trade-off é claro: Arrays são superiores para leitura "
        "aleatória intensiva; Listas são mais eficientes para inserções/remoções "
        "frequentes em posições intermediárias, especialmente quando o ponteiro de "
        "inserção já é conhecido."
    ),
    (1, 1): (
        # Intermediário: correto mas superficial, sem Big-O explícito na maioria dos casos
        "Arrays guardam os dados de forma organizada na memória, um do lado do outro, "
        "o que permite acessar qualquer elemento diretamente pelo índice. "
        "Listas Encadeadas funcionam de forma diferente: cada elemento aponta para o "
        "próximo, então para acessar o elemento k é preciso percorrer desde o início. "
        "Quando se insere ou remove no meio de uma Array, os outros elementos precisam "
        "se mover para abrir ou fechar espaço. Na Lista, basta mudar os ponteiros, "
        "sem mover dados. No geral, Arrays são boas para acesso rápido e Listas "
        "para quando se modifica muito a estrutura."
    ),
    (1, 2): (
        # Fraco: erra acesso em Array (diz O(n)), diz Lista aloca memória de uma vez,
        # inverte complexidade de inserção/remoção
        "Arrays são estruturas onde os dados ficam armazenados em ordem sequencial. "
        "O acesso é feito de forma sequencial, então para chegar ao elemento k é preciso "
        "percorrer todos os anteriores, custando O(n). "
        "Listas Encadeadas alocam toda a memória necessária de uma vez no início do "
        "programa, e os nós ficam conectados. O acesso a qualquer elemento é direto "
        "pelo índice, sendo O(1). "
        "Para inserir ou remover no meio de uma Array, o custo é O(1) porque os "
        "elementos não precisam ser movidos. Já na Lista, o custo é O(n²) porque "
        "é necessário reorganizar todos os ponteiros da estrutura inteira."
    ),
    (1, 3): (
        # Fora do tema: resposta sobre fotossíntese
        "A fotossíntese é o processo pelo qual as plantas convertem luz solar em "
        "energia química. Ocorre nos cloroplastos, onde a clorofila absorve luz e usa "
        "CO2 e água para produzir glicose e oxigênio. É fundamental para a cadeia "
        "alimentar e para a manutenção do oxigênio na atmosfera terrestre."
    ),

    # ── Q2: Merge Sort vs Quick Sort ─────────────────────────────────────────
    (2, 0): (
        # Excelente
        "Merge Sort é estável: elementos com chaves iguais preservam sua ordem relativa "
        "original após a ordenação. Sua complexidade de tempo é O(n log n) no melhor, "
        "médio e pior caso, garantindo desempenho consistente independente da entrada. "
        "O custo de espaço é O(n) auxiliar, pois requer um array temporário na mesclagem. "
        "Quick Sort é instável: a posição relativa de elementos iguais pode mudar após "
        "a partição. Sua complexidade média é O(n log n), mas degenera para O(n²) no "
        "pior caso (pivô sempre extremo, ex.: array já ordenado sem aleatorização do pivô). "
        "O espaço auxiliar é O(log n) em média (pilha de recursão). "
        "Implicações práticas: Merge Sort é preferível quando estabilidade é requisito "
        "(ex.: ordenar registros por múltiplos campos mantendo sub-ordenação prévia) ou "
        "quando desempenho garantido é crítico. Quick Sort é preferível com memória "
        "limitada e dados sem padrões degenerativos, sendo geralmente mais rápido na "
        "prática por menor constante multiplicativa e melhor localidade de cache."
    ),
    (2, 1): (
        # Intermediário: correto mas raso, sem explicar por que QS degenera,
        # sem discutir implicações práticas reais
        "Merge Sort é um algoritmo estável, enquanto Quick Sort não é estável. "
        "O Merge Sort tem complexidade O(n log n) mesmo no pior caso, o que o torna "
        "mais previsível. O Quick Sort pode chegar a O(n²) no pior caso, mas em média "
        "é O(n log n). Em relação ao espaço, o Merge Sort precisa de memória extra "
        "proporcional ao tamanho do array, enquanto Quick Sort usa menos memória "
        "adicional. Para escolher entre eles, deve-se considerar se a estabilidade "
        "importa e quanto de memória está disponível no sistema."
    ),
    (2, 2): (
        # Fraco: inverte estabilidade (MS instável, QS estável), erra complexidade
        # (diz MS O(n²) e QS O(n log n) em todos casos)
        "Merge Sort é um algoritmo instável que divide o array ao meio e depois junta "
        "as partes. Sua complexidade no pior caso é O(n²) e usa muito espaço de memória. "
        "Quick Sort é estável e funciona escolhendo um pivô para separar elementos "
        "menores e maiores. Tem complexidade O(n log n) em todos os casos e usa pouco "
        "espaço. Por isso o Quick Sort é sempre mais eficiente que o Merge Sort, "
        "especialmente em arrays já ordenados onde o Merge Sort fica muito lento."
    ),
    (2, 3): (
        # Fora do tema: resposta sobre culinária
        "Fritar e cozinhar são técnicas culinárias muito diferentes. Fritar usa gordura "
        "quente para criar uma crosta crocante por fora enquanto mantém o interior úmido. "
        "Cozinhar em água transmite calor por condução e é indicado para legumes e massas. "
        "A escolha depende do ingrediente e do resultado desejado em termos de textura "
        "e sabor do prato final."
    ),

    # ── Q3: BST Balanceada vs Não Balanceada ─────────────────────────────────
    (3, 0): (
        # Excelente
        "Uma BST é definida pela propriedade de ordenação: todo nó à esquerda tem chave "
        "menor e à direita maior que o nó pai. Em uma BST balanceada (ex.: AVL, Red-Black), "
        "a diferença de altura entre as subárvores esquerda e direita de qualquer nó é "
        "limitada (fator de balanceamento ≤ 1 em AVL), garantindo altura O(log n). "
        "Em uma BST não balanceada, inserções em ordem crescente ou decrescente degeram "
        "a estrutura em uma lista linear, com altura O(n). "
        "A principal vantagem da BST balanceada é garantir complexidade O(log n) para "
        "busca, inserção e remoção em todos os cenários (melhor e pior caso). "
        "Na não balanceada: melhor caso O(log n) (árvore naturalmente equilibrada pela "
        "ordem aleatória das inserções) e pior caso O(n) (árvore degenerada). Esse "
        "comportamento imprevisível é crítico em aplicações com requisitos de latência "
        "garantida, justificando o overhead de rebalanceamento das árvores autobalanceadas."
    ),
    (3, 1): (
        # Intermediário: correto mas sem profundidade — não cita exemplos de AVL/Red-Black,
        # não explica fator de balanceamento, não discute custo do rebalanceamento
        "Uma BST balanceada mantém a altura da árvore pequena, próxima de O(log n), "
        "enquanto uma não balanceada pode crescer de forma descontrolada dependendo "
        "da ordem de inserção dos elementos. A vantagem principal da balanceada é que "
        "busca, inserção e remoção ficam em O(log n) sempre. Na não balanceada, essas "
        "operações podem chegar a O(n) no pior caso, quando a árvore vira praticamente "
        "uma lista encadeada. No melhor caso, ambas conseguem O(log n) se os elementos "
        "forem inseridos de forma bem distribuída."
    ),
    (3, 2): (
        # Fraco: confunde propriedade da BST (diz filho sempre maior que pai — errado),
        # inventa O(1) para busca na balanceada e O(n²) na não balanceada
        "Uma BST balanceada tem os nós organizados de forma que cada filho seja sempre "
        "maior que o pai, tornando a estrutura simétrica e eficiente. "
        "A BST não balanceada pode ter os nós em qualquer posição, o que a torna "
        "desorganizada e difícil de percorrer. "
        "A vantagem da balanceada é que o acesso é sempre muito rápido. "
        "A complexidade de busca na balanceada é O(1) no melhor e pior caso. "
        "Na não balanceada, o melhor caso é O(log n) e o pior caso é O(n²), pois "
        "é necessário percorrer toda a estrutura reorganizando os nós durante a busca."
    ),
    (3, 3): (
        # Fora do tema: resposta sobre redes neurais
        "Redes neurais artificiais são modelos computacionais inspirados no cérebro "
        "humano. São compostas por camadas de neurônios artificiais que processam "
        "informações de entrada. As redes profundas (deep learning) possuem múltiplas "
        "camadas ocultas e são utilizadas em reconhecimento de imagens, processamento "
        "de linguagem natural e outros problemas de aprendizado de máquina supervisionado."
    ),
}

print(f"✅ Dados carregados: {len(QUESTIONS)} questões, {len(CRITERIAS)} critérios, {len(STUDENTS)} alunos, {len(ANSWERS)} respostas")


✅ Dados carregados: 3 questões, 5 critérios, 4 alunos, 12 respostas


## 2. Setup — Registro, Login, Turma e Alunos

### 2.1 Registro do professor

In [19]:
TEACHER = {
    "first_name": "Professor",
    "last_name": "TCC",
    "email": f"professor.tcc.{int(time.time())}@example.com",
    "password": "SenhaSegura@123",
    "user_type": "teacher",
}

r = api("post", "/users/register", json=TEACHER)
print(f"POST /users/register → {r.status_code}")

try:
    teacher_data = r.json()
except Exception:
    print("Resposta não-JSON:", r.text[:1000])
    r.raise_for_status()
if r.status_code >= 400:
    print("Resposta de erro:", r.status_code, r.text[:1000])
    r.raise_for_status()
# Extrair uuid de formatos comuns: top-level 'uuid' ou 'data' -> 'uuid'
teacher_uuid = teacher_data.get("uuid") or (teacher_data.get("data") or {}).get("uuid")
if not teacher_uuid:
    print("Resposta JSON (completa):")
    print(json.dumps(teacher_data, ensure_ascii=False, indent=2))
    raise KeyError("campo 'uuid' ausente na resposta do registro.")
state["teacher_uuid"] = teacher_uuid
print(f"✅ Professor registrado: {state['teacher_uuid']}")

POST /users/register → 201
✅ Professor registrado: 5ca48b5e-37bc-4949-894d-7b6ced5369bd


### 2.2 Login

In [20]:
r = api("post", "/auth/login", json={
    "email": TEACHER["email"],
    "password": TEACHER["password"],
})
r.raise_for_status()
state["token"] = r.json()["access_token"]
print(f"✅ Login OK — token obtido")

✅ Login OK — token obtido


### 2.3 Criar turma

In [21]:
r = api("post", "/classes", json={
    "name": "Algoritmos e Estrutura de Dados — TCC Experimento",
    "description": "Turma experimental para validação do sistema de correção automática",
    "year": 2026,
    "semester": 1,
})
r.raise_for_status()
state["class_uuid"] = r.json()["uuid"]
print(f"✅ Turma criada: {state['class_uuid']}")

✅ Turma criada: 58bbb3ee-2ebf-47e2-93f0-8f5f9b2dce8c


### 2.4 Adicionar alunos

In [22]:
r = api("post", f"/classes/{state['class_uuid']}/students", json={
    "students": [
        {"full_name": s["name"], "email": s["email"]}
        for s in STUDENTS
    ]
})
print(f"POST /classes/.../students → {r.status_code}")
try:
    students_resp = r.json()
    print("Resposta JSON (students_resp):")
    print(json.dumps(students_resp, ensure_ascii=False, indent=2)[:2000])
except Exception:
    print("Resposta não-JSON:", r.text[:1000])
    r.raise_for_status()
if r.status_code >= 400:
    print("Resposta de erro:", r.status_code, r.text[:1000])
    r.raise_for_status()
# Extrair alunos de formatos possíveis: top-level 'students'|'enrolled' ou 'details'
details = students_resp.get("details", {}) if isinstance(students_resp, dict) else {}
enrolled = students_resp.get("students") or students_resp.get("enrolled") or details.get("enrolled") or []
created = details.get("created", []) or []
# Combinar mantendo ordem e evitando duplicatas simples
all_new = list(enrolled) + [c for c in created if c not in enrolled]
if not isinstance(all_new, list):
    print("Formato inesperado na resposta (esperado lista):")
    print(json.dumps(students_resp, ensure_ascii=False, indent=2))
    raise KeyError("Lista de alunos não encontrada na resposta")
# Extrair uuids tolerando 'uuid' ou 'id'
state["student_uuids"] = [s.get("uuid") or s.get("id") for s in all_new]
for i, s in enumerate(all_new):
    uuid = s.get("uuid") or s.get("id") or "<sem uuid>"
    profile = STUDENTS[i]['profile'] if i < len(STUDENTS) else '<desconhecido>'
    print(f"  {profile:>10} → {uuid}")
print(f"✅ {len(all_new)} alunos adicionados")
# Salvar CSV com alunos criados / matriculados
try:
    df_students = pd.DataFrame(all_new)
    if not df_students.empty:
        cols = [c for c in ["uuid", "id", "full_name", "email"] if c in df_students.columns]
        df_students = df_students[cols] if cols else df_students
        save_csv(df_students, "students_added")
except Exception as e:
    print("Erro ao salvar CSV de alunos:", e)

POST /classes/.../students → 200
Resposta JSON (students_resp):
{
  "class_uuid": "58bbb3ee-2ebf-47e2-93f0-8f5f9b2dce8c",
  "class_name": "Algoritmos e Estrutura de Dados — TCC Experimento",
  "summary": {
    "total_requested": 4,
    "students_enrolled": 4,
    "students_already_enrolled": 0,
    "new_students_created": 0,
    "errors": 0
  },
  "details": {
    "enrolled": [
      {
        "uuid": "d83c77f6-37a2-46c9-8723-ba76bb44b5ac",
        "full_name": "Aluno 1 (Excelente)",
        "email": "aluno1.excelente@example.com"
      },
      {
        "uuid": "9b6aabce-8b4d-49f4-908a-b0777cacb7c8",
        "full_name": "Aluno 2 (Intermediário)",
        "email": "aluno2.adequado@example.com"
      },
      {
        "uuid": "1577ae18-04c1-415c-ba1f-c1deda50b60a",
        "full_name": "Aluno 3 (Fraco)",
        "email": "aluno3.fraco@example.com"
      },
      {
        "uuid": "9e493d13-3a78-4d28-ab35-254bd144dd5d",
        "full_name": "Aluno 4 (Fora do Tema)",
        "email": "

### 2.5 Buscar UUIDs dos critérios do banco

In [23]:
r = api("get", "/grading-criteria", params={"active_only": True, "limit": 100})
r.raise_for_status()
available_criteria = r.json()

# Mapear nome → uuid
criteria_name_to_uuid = {c["name"]: c["uuid"] for c in available_criteria}
print(f"  ✅ {len(available_criteria)} critérios disponíveis no banco:")
for c in available_criteria:
    print(f"     • {c['name']} → {c['uuid']}")

# Enriquecer CRITERIAS com os UUIDs do banco (match por nome)
assert len(available_criteria) > 0, "❌ Nenhum critério retornado — verifique se as migrations rodaram (alembic upgrade head)"

for c in CRITERIAS:
    uuid = criteria_name_to_uuid.get(c["name"])
    if not uuid:
        raise ValueError(f"Critério '{c['name']}' não encontrado no banco. Disponíveis: {list(criteria_name_to_uuid.keys())}")
    c["uuid"] = uuid

print(f"✅ UUIDs resolvidos para {len(CRITERIAS)} critérios")
for c in CRITERIAS:
    print(f"     • {c['name']} → {c['uuid']}")

  ✅ 28 critérios disponíveis no banco:
     • Adequação ao gênero solicitado → f641b04e-feb3-47e0-8c89-d0f9ced43ebe
     • Análise crítica e profundidade → 2e892085-9a73-4312-9413-a87ad08fb41f
     • Argumentação e justificativa → 11094edf-abb4-4fc1-8516-40c2d0ea4e7d
     • Atendimento ao enunciado → 49d23224-cb2e-4583-a16e-0d89317c04a5
     • Clareza e objetividade → 0ead5c4c-ff20-4a11-bd5c-a7d9b97d746d
     • Coerência e coesão textual → 9613426b-8c64-4bc9-a8ee-30420641efc6
     • Completude da resposta → 9166f34c-6b3d-4eae-a1ee-bd1816086f71
     • Conclusão / encaminhamento → ba46ad3d-d578-41e1-bb35-1fca808906ed
     • Consideração de contrapartes → 1b46e226-f2e0-45d3-95c1-304bdd4fdf35
     • Contextualização e repertório → 5638e006-10f0-4986-a5e1-b969f5c9f616
     • Correção factual → 2129cfb6-9a63-4c26-a104-8c07adcba72f
     • Correção técnica → efaba390-003f-4002-926f-ab6b24b155eb
     • Estrutura e organização → 6209379c-0dae-4d84-9515-02c67b4988ee
     • Ética, segurança e conf

## DEF run_experiment

In [24]:
def run_experiment(label, upload_pdf=True, poll_interval=5, max_wait=300):    
    
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"\n{'='*70}")
    print(f"  {label} — início {ts}")
    print(f"  RAG: {'SIM' if upload_pdf else 'NÃO'}")
    print(f"{'='*70}\n")

    # ── Criar prova ───────────────────────────────────────────────────────
    now = datetime.now()
    r = api("post", "/exams", json={
        "title": f"{label} — {now.strftime('%Y%m%d_%H%M%S')}",
        "description": f"Experimento TCC: {label}",
        "class_uuid": state["class_uuid"],
        "starts_at": now.isoformat(),
        "ends_at": (now + timedelta(hours=2)).isoformat(),
    })
    r.raise_for_status()
    exam_uuid = r.json()["uuid"]
    print(f"  📝 Prova criada: {exam_uuid}")

    # ── Criar critérios de avaliação (exam-criteria) a partir de CRITERIAS ──
    # Garante que todos os UUIDs foram resolvidos pela célula 2.5
    missing = [c["name"] for c in CRITERIAS if not c.get("uuid")]
    assert not missing, f"❌ UUIDs não resolvidos para: {missing}. Execute a célula 2.5 primeiro."

    criteria_added = 0
    for c in CRITERIAS:
        r2 = api("post", "/exam-criteria", json={
            "exam_uuid": exam_uuid,
            "criteria_uuid": c["uuid"],
            "weight": c["weight"],
            "max_points": c["max_points"],
        })
        r2.raise_for_status()
        criteria_added += 1
    print(f"  ✅ {criteria_added} critérios configurados")

    # ── Criar questões ────────────────────────────────────────────────────
    question_uuids = []
    for q in QUESTIONS:
        r = api("post", "/exam-questions", json={
            "exam_uuid": exam_uuid,
            "statement": q["statement"],
            "question_order": q["order"],
            "points": q["points"],
        })
        r.raise_for_status()
        question_uuids.append(r.json()["uuid"])
    print(f"  📋 {len(question_uuids)} questões criadas")

    # Mapear UUID da questão criada → Q-id (Q1, Q2, Q3)
    uuid_to_qid = {question_uuids[i]: QUESTIONS[i]["id"] for i in range(len(QUESTIONS))}

    # ── Submeter respostas ────────────────────────────────────────────────
    answer_count = 0
    for (q_order, s_idx), answer_text in ANSWERS.items():
        r = api("post", "/student-answers", json={
            "exam_uuid": exam_uuid,
            "question_uuid": question_uuids[q_order - 1],
            "student_uuid": state["student_uuids"][s_idx],
            "answer_text": answer_text,
        })
        r.raise_for_status()
        answer_count += 1
    print(f"  ✏️  {answer_count} respostas submetidas")

    # ── Upload PDF (RAG) ──────────────────────────────────────────────────
    if upload_pdf:
        with open(PDF_PATH, "rb") as f:
            r = api("post", f"/attachments/upload",
                    params={"exam_uuid": exam_uuid},
                    files={"file": (PDF_PATH.name, f, "application/pdf")})
        r.raise_for_status()
        print(f"  📄 PDF enviado para RAG")
    else:
        print(f"  📄 Sem PDF — condição sem RAG")

    # ── Publicar (dispara correção em background) ─────────────────────────
    r = api("post", f"/exams/{exam_uuid}/publish")
    r.raise_for_status()
    print(f"  🚀 Prova publicada — correção iniciada")

    # ── Polling até GRADED ────────────────────────────────────────────────
    start = time.time()
    while time.time() - start < max_wait:
        time.sleep(poll_interval)
        r = api("get", f"/exams/{exam_uuid}")
        r.raise_for_status()
        status = r.json().get("status", "")
        elapsed = int(time.time() - start)
        print(f"  ⏳ {elapsed}s — status: {status}        ", end="\r")
        if status in ("GRADED", "WARNING"):
            print(f"  ✅ Correção finalizada em {elapsed}s — status: {status}")
            break
    else:
        print(f"  ⚠️ Timeout ({max_wait}s) — verificar manualmente")

    # ── Coletar resultados via review ─────────────────────────────────────
    r = api("get", f"/reviews/exams/{exam_uuid}")
    r.raise_for_status()
    review = r.json()

    results = []
    rag_rows = []
    for q_data in review.get("questions", []):
        q_uuid = str(q_data.get("question_uuid", q_data.get("uuid", "")))
        q_id = uuid_to_qid.get(q_uuid, "?")

        for ctx in q_data.get("rag_contexts", []):
            rag_rows.append({
                "exam_uuid": exam_uuid,
                "label": label,
                "rag": upload_pdf,
                "question_id": q_id,
                "question_uuid": q_uuid,
                "source_document": ctx.get("source_document"),
                "page_number": ctx.get("page_number"),
                "relevance_score": ctx.get("relevance_score"),
                "chunk_index": ctx.get("chunk_index"),
                "content": ctx.get("content", ""),
            })

        for ans in q_data.get("student_answers", q_data.get("answers", [])):
            results.append({
                "exam_uuid": exam_uuid,
                "label": label,
                "rag": upload_pdf,
                "question_id": q_id,
                "question": q_data.get("statement", "")[:60],
                "question_uuid": q_uuid,
                "student_name": ans.get("student_name", ""),
                "profile": PROFILE_MAP.get(ans.get("student_name", ""), "?"),
                "student_uuid": ans.get("student_uuid"),
                "score": ans.get("score"),
                "c1_score": ans.get("c1_score"),
                "c2_score": ans.get("c2_score"),
                "arbiter_score": ans.get("arbiter_score"),
                "divergence_detected": ans.get("divergence_detected", False),
                "divergence_value": ans.get("divergence_value"),
                "consensus_method": ans.get("consensus_method"),
                "feedback": ans.get("feedback", ""),
            })

    df = pd.DataFrame(results)
    df_rag = pd.DataFrame(rag_rows)
    print(f"\n  📊 {len(results)} resultados coletados")
    if len(df) > 0:
        print(f"  📈 Média geral: {df['score'].mean():.2f}")
        n_div = df["divergence_detected"].sum()
        if n_div > 0:
            print(f"  ⚠️  {n_div} divergências detectadas (árbitro acionado)")

    return {
        "label": label,
        "exam_uuid": exam_uuid,
        "rag": upload_pdf,
        "df": df,
        "df_rag": df_rag,
        "raw_review": review,
    }

## DEF run_qa4_experiment

In [25]:
def run_qa4_experiment(run_label, poll_interval=5, max_wait=300):
    """
    QA4 — Estabilidade: executa correção apenas do subconjunto A_rep=4.
    Cria uma prova com UMA questão (Q2, mais discriminatória) e as 4 respostas
    sintéticas (uma por nível: Excelente, Intermediário, Fraco, Fora do tema).
    Mantém RAG ativo e mesmos critérios de run_experiment.
    """
    ts = datetime.now().strftime('%H:%M:%S')
    print(f"  [{run_label}] início {ts}")

    # Garante UUIDs resolvidos
    missing = [c['name'] for c in CRITERIAS if not c.get('uuid')]
    assert not missing, f'❌ Execute a célula 2.5 primeiro. Faltam: {missing}'

    # Questão de referência para QA4 (Q2 — mais discriminatória entre perfis)
    QA4_QUESTION = QUESTIONS[1]  # Q2
    QA4_Q_ORDER = QA4_QUESTION['order']  # 2

    # Criar prova
    now = datetime.now()
    r = api('post', '/exams', json={
        'title': f'QA4 {run_label} — {now.strftime("%Y%m%d_%H%M%S")}',
        'description': 'QA4 Estabilidade — subconjunto A_rep=4',
        'class_uuid': state['class_uuid'],
        'starts_at': now.isoformat(),
        'ends_at': (now + timedelta(hours=2)).isoformat(),
    })
    r.raise_for_status()
    exam_uuid = r.json()['uuid']

    # Critérios
    for c in CRITERIAS:
        r2 = api('post', '/exam-criteria', json={
            'exam_uuid': exam_uuid,
            'criteria_uuid': c['uuid'],
            'weight': c['weight'],
            'max_points': c['max_points'],
        })
        r2.raise_for_status()

    # Criar apenas Q2
    r = api('post', '/exam-questions', json={
        'exam_uuid': exam_uuid,
        'statement': QA4_QUESTION['statement'],
        'question_order': 1,
        'points': QA4_QUESTION['points'],
    })
    r.raise_for_status()
    q_uuid = r.json()['uuid']

    # Submeter as 4 respostas de Q2 (s_idx 0..3)
    for s_idx in range(len(STUDENTS)):
        answer_text = ANSWERS[(QA4_Q_ORDER, s_idx)]
        r = api('post', '/student-answers', json={
            'exam_uuid': exam_uuid,
            'question_uuid': q_uuid,
            'student_uuid': state['student_uuids'][s_idx],
            'answer_text': answer_text,
        })
        r.raise_for_status()

    # Upload PDF (RAG sempre ativo no QA4)
    with open(PDF_PATH, 'rb') as f:
        r = api('post', '/attachments/upload',
                params={'exam_uuid': exam_uuid},
                files={'file': (PDF_PATH.name, f, 'application/pdf')})
    r.raise_for_status()

    # Publicar
    r = api('post', f'/exams/{exam_uuid}/publish')
    r.raise_for_status()

    # Polling
    import time as _time
    start = _time.time()
    while _time.time() - start < max_wait:
        _time.sleep(poll_interval)
        status = api('get', f'/exams/{exam_uuid}').json().get('status', '')
        elapsed = int(_time.time() - start)
        print(f'  ⏳ {elapsed}s — {status}        ', end='')
        if status in ('GRADED', 'WARNING'):
            print(f'  ✅ {elapsed}s — {status}')
            break

    # Coletar resultados
    review = api('get', f'/reviews/exams/{exam_uuid}').json()
    results = []
    rag_rows = []
    for q_data in review.get('questions', []):
        for ctx in q_data.get('rag_contexts', []):
            rag_rows.append({
                'exam_uuid': exam_uuid,
                'run_label': run_label,
                'question_id': 'Q2',
                'question_uuid': str(q_data.get('question_uuid', '')),
                'source_document': ctx.get('source_document'),
                'page_number': ctx.get('page_number'),
                'relevance_score': ctx.get('relevance_score'),
                'chunk_index': ctx.get('chunk_index'),
                'content': ctx.get('content', ''),
            })
        for ans in q_data.get('student_answers', q_data.get('answers', [])):
            s_name = ans.get('student_name', '')
            results.append({
                'exam_uuid': exam_uuid,
                'run_label': run_label,
                'question_id': 'Q2',
                'student_name': s_name,
                'profile': PROFILE_MAP.get(s_name, '?'),
                'student_uuid': ans.get('student_uuid'),
                'score': ans.get('score'),
                'c1_score': ans.get('c1_score'),
                'c2_score': ans.get('c2_score'),
                'divergence_detected': ans.get('divergence_detected', False),
                'divergence_value': ans.get('divergence_value'),
                'consensus_method': ans.get('consensus_method'),
            })

    return {'run_label': run_label, 'exam_uuid': exam_uuid, 'df': pd.DataFrame(results), 'df_rag': pd.DataFrame(rag_rows), 'raw_review': review}


## 4. Condição A — Com RAG

In [26]:
cond_a = run_experiment("Condição A (com RAG)", upload_pdf=True)


  Condição A (com RAG) — início 16:38:34
  RAG: SIM

  📝 Prova criada: 7b49f0e2-fcb1-43e4-8d20-7891e9e62071
  ✅ 5 critérios configurados
  📋 3 questões criadas
  ✏️  12 respostas submetidas
  📄 PDF enviado para RAG
  🚀 Prova publicada — correção iniciada
  ✅ Correção finalizada em 155s — status: GRADED

  📊 12 resultados coletados
  📈 Média geral: 5.09


In [27]:
# Tabela detalhada + salva CSV
cols = ["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected", "consensus_method"]
display(cond_a["df"][cols])

save_csv(cond_a["df"], "cond_A")
if len(cond_a["df_rag"]) > 0:
    save_csv(cond_a["df_rag"], "cond_A_rag")

,question_id,profile,c1_score,c2_score,arbiter_score,score,divergence_detected,consensus_method
0,Q1,Intermediário,7.59,8.65,None,8.65,False,mean_2
1,Q1,Fraca,0.92,0.00,None,0.00,False,mean_2
2,Q1,Fora tema,0.00,0.00,None,0.00,False,mean_2
3,Q1,Excelente,10.00,10.00,None,10.00,False,mean_2
4,Q2,Intermediário,8.29,9.47,None,9.47,False,mean_2
5,Q2,Fraca,1.89,2.55,None,2.55,False,mean_2
6,Q2,Fora tema,0.00,0.00,None,0.00,False,mean_2
7,Q2,Excelente,10.00,10.00,None,10.00,False,mean_2
8,Q3,Excelente,10.00,10.00,None,10.00,False,mean_2
9,Q3,Intermediário,10.00,9.80,None,9.80,False,mean_2


  💾 ..\notebooks\results\cond_A_20260328_163833.csv
  💾 ..\notebooks\results\cond_A_rag_20260328_163833.csv


## 5. Condição B — Sem RAG

In [28]:
cond_b = run_experiment("Condição B (sem RAG)", upload_pdf=False)


  Condição B (sem RAG) — início 16:41:46
  RAG: NÃO

  📝 Prova criada: 48304f9b-1754-4cbd-b973-4a39a51a20a2
  ✅ 5 critérios configurados
  📋 3 questões criadas
  ✏️  12 respostas submetidas
  📄 Sem PDF — condição sem RAG
  🚀 Prova publicada — correção iniciada
  ✅ Correção finalizada em 140s — status: GRADED

  📊 12 resultados coletados
  📈 Média geral: 4.86


In [29]:
# Tabela detalhada + salva CSV
cols = ["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected", "consensus_method"]
display(cond_b["df"][cols])

save_csv(cond_b["df"], "cond_B")
if len(cond_b["df_rag"]) > 0:
    save_csv(cond_b["df_rag"], "cond_B_rag")

,question_id,profile,c1_score,c2_score,arbiter_score,score,divergence_detected,consensus_method
0,Q1,Intermediário,6.24,6.73,None,6.73,False,mean_2
1,Q1,Fraca,0.51,0.00,None,0.00,False,mean_2
2,Q1,Fora tema,0.00,0.00,None,0.00,False,mean_2
3,Q1,Excelente,10.00,10.00,None,10.00,False,mean_2
4,Q2,Excelente,10.00,10.00,None,10.00,False,mean_2
5,Q2,Intermediário,8.88,8.88,None,8.88,False,mean_2
6,Q2,Fraca,2.55,1.63,None,1.63,False,mean_2
7,Q2,Fora tema,0.00,0.00,None,0.00,False,mean_2
8,Q3,Intermediário,9.80,9.80,None,9.80,False,mean_2
9,Q3,Fraca,0.00,1.22,None,1.22,False,mean_2


  💾 ..\notebooks\results\cond_B_20260328_163833.csv


## 6. Comparação A vs B

In [30]:
# Comparação pareada A vs B com detalhes dos corretores
a = cond_a["df"][["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected"]].copy()
b = cond_b["df"][["question_id", "profile", "c1_score", "c2_score", "arbiter_score", "score", "divergence_detected"]].copy()

a.columns = ["question_id", "profile", "A_c1", "A_c2", "A_arb", "A_final", "A_div"]
b.columns = ["question_id", "profile", "B_c1", "B_c2", "B_arb", "B_final", "B_div"]

comparison = a.merge(b, on=["question_id", "profile"])
comparison["Delta"] = comparison["A_final"] - comparison["B_final"]

display(comparison)

save_csv(comparison, "comparacao_A_vs_B")

,question_id,profile,A_c1,A_c2,A_arb,A_final,A_div,B_c1,B_c2,B_arb,B_final,B_div,Delta
0,Q1,Intermediário,7.59,8.65,None,8.65,False,6.24,6.73,None,6.73,False,1.92
1,Q1,Fraca,0.92,0.00,None,0.00,False,0.51,0.00,None,0.00,False,0.00
2,Q1,Fora tema,0.00,0.00,None,0.00,False,0.00,0.00,None,0.00,False,0.00
3,Q1,Excelente,10.00,10.00,None,10.00,False,10.00,10.00,None,10.00,False,0.00
4,Q2,Intermediário,8.29,9.47,None,9.47,False,8.88,8.88,None,8.88,False,0.59
5,Q2,Fraca,1.89,2.55,None,2.55,False,2.55,1.63,None,1.63,False,0.92
6,Q2,Fora tema,0.00,0.00,None,0.00,False,0.00,0.00,None,0.00,False,0.00
7,Q2,Excelente,10.00,10.00,None,10.00,False,10.00,10.00,None,10.00,False,0.00
8,Q3,Excelente,10.00,10.00,None,10.00,False,10.00,10.00,None,10.00,False,0.00
9,Q3,Intermediário,10.00,9.80,None,9.80,False,9.80,9.80,None,9.80,False,0.00


  💾 ..\notebooks\results\comparacao_A_vs_B_20260328_163833.csv


WindowsPath('../notebooks/results/comparacao_A_vs_B_20260328_163833.csv')

In [31]:
# Resumo por nível de qualidade
summary = comparison.groupby("profile").agg(
    A_final=("A_final", "mean"),
    B_final=("B_final", "mean"),
    Delta=("Delta", "mean"),
).round(2)

print("Média por nível de qualidade:")
display(summary)

save_csv(summary.reset_index(), "resumo_por_nivel")

Média por nível de qualidade:


,A_final,B_final,Delta
profile,,,
Excelente,10.00,10.00,0.00
Fora tema,0.00,0.00,0.00
Fraca,1.07,0.95,0.12
Intermediário,9.31,8.47,0.84


  💾 ..\notebooks\results\resumo_por_nivel_20260328_163833.csv


WindowsPath('../notebooks/results/resumo_por_nivel_20260328_163833.csv')

# QA4 — Estabilidade: R=3 repetições, A_rep=4 respostas (uma por nível de qualidade)

In [32]:
# QA4 — Estabilidade: R=3 repetições, A_rep=4 respostas (uma por nível de qualidade)
# Conforme metodologia do TCC: A_rep=4 perfis, R=3 repetições.
# Cada run cria uma prova nova com apenas Q2 e as 4 respostas sintéticas (RAG ativo).

QA4_RUNS = 3  # R=3 conforme metodologia

qa4_results = []
for i in range(QA4_RUNS):
    result = run_qa4_experiment(f"Run {i+1}/{QA4_RUNS}")
    result["df"]["run"] = i + 1
    qa4_results.append(result)
    print(f"{chr(9472)*70}")

print(f"✅ QA4 completo — {QA4_RUNS} execuções realizadas (R={QA4_RUNS}, A_rep=4 níveis)")

  [Run 1/3] início 16:44:07
  ⏳ 5s — PUBLISHED          ⏳ 10s — PUBLISHED          ⏳ 15s — PUBLISHED          ⏳ 20s — PUBLISHED          ⏳ 25s — PUBLISHED          ⏳ 30s — PUBLISHED          ⏳ 35s — PUBLISHED          ⏳ 40s — PUBLISHED          ⏳ 45s — PUBLISHED          ⏳ 50s — PUBLISHED          ⏳ 55s — GRADED          ✅ 55s — GRADED
──────────────────────────────────────────────────────────────────────
  [Run 2/3] início 16:45:39
  ⏳ 5s — PUBLISHED          ⏳ 10s — PUBLISHED          ⏳ 15s — PUBLISHED          ⏳ 20s — PUBLISHED          ⏳ 25s — PUBLISHED          ⏳ 30s — PUBLISHED          ⏳ 35s — PUBLISHED          ⏳ 40s — PUBLISHED          ⏳ 45s — PUBLISHED          ⏳ 50s — PUBLISHED          ⏳ 55s — GRADED          ✅ 55s — GRADED
──────────────────────────────────────────────────────────────────────
  [Run 3/3] início 16:47:10
  ⏳ 5s — PUBLISHED          ⏳ 10s — PUBLISHED          ⏳ 15s — PUBLISHED          ⏳ 20s — PUBLISHED          ⏳ 25s — PUBLISHED          ⏳ 30s — PUBLISHED 

In [33]:
# Consolidar todas as runs do QA4
all_qa4 = pd.concat([r["df"] for r in qa4_results], ignore_index=True)

# Salva CSV com contextos RAG do QA4
all_qa4_rag = pd.concat([r["df_rag"] for r in qa4_results if len(r["df_rag"]) > 0], ignore_index=True)
if len(all_qa4_rag) > 0:
    save_csv(all_qa4_rag, "qa4_rag")

# Salva CSV com todas as runs brutas
save_csv(all_qa4, "qa4_todas_runs")

# ── Estatísticas por caso (question_id × profile) ────────────────────────
qa4_stats = all_qa4.groupby(["question_id", "profile"]).agg(
    mean_score=("score", "mean"),
    std_score=("score", "std"),
    min_score=("score", "min"),
    max_score=("score", "max"),
    var_score=("score", "var"),
    # Variação absoluta = max - min por caso (métrica de sucesso do QA4 no TCC)
    var_abs=("score", lambda x: x.max() - x.min()),
    mean_c1=("c1_score", "mean"),
    mean_c2=("c2_score", "mean"),
    n_divergencias=("divergence_detected", "sum"),
).round(2)

# Critério de sucesso QA4: var_abs <= 1.0 em todos os casos
qa4_aprovados = (qa4_stats["var_abs"] <= 1.0).all()
print(f"Variância média:              {qa4_stats['var_score'].mean():.2f}")
print(f"Desvio padrão médio:          {qa4_stats['std_score'].mean():.2f}")
print(f"Variação absoluta máxima:     {qa4_stats['var_abs'].max():.2f}")
print(f"QA4 aprovado (var_abs<=1.0):  {'✅ SIM' if qa4_aprovados else '❌ NÃO'}")
print()
display(qa4_stats)

save_csv(qa4_stats.reset_index(), "qa4_estatisticas")

  💾 ..\notebooks\results\qa4_rag_20260328_163833.csv
  💾 ..\notebooks\results\qa4_todas_runs_20260328_163833.csv
Variância média:              0.20
Desvio padrão médio:          0.30
Variação absoluta máxima:     1.37
QA4 aprovado (var_abs<=1.0):  ❌ NÃO



mean_score  std_score  min_score  max_score  \
question_id profile                                                      
Q2          Excelente           10.00       0.00      10.00      10.00   
            Fora tema            0.00       0.00       0.00       0.00   
            Fraca                1.37       0.75       0.51       1.88   
            Intermediário        9.33       0.47       9.04       9.88   

                           var_score  var_abs  mean_c1  mean_c2  \
question_id profile                                               
Q2          Excelente           0.00     0.00    10.00    10.00   
            Fora tema           0.00     0.00     0.00     0.00   
            Fraca               0.56     1.37     2.03     1.37   
            Intermediário       0.22     0.84     9.48     9.33   

                           n_divergencias  
question_id profile                        
Q2          Excelente                   0  
            Fora tema                   0  
            Fraca                       0  
            Intermediário               0

  💾 ..\notebooks\results\qa4_estatisticas_20260328_163833.csv


WindowsPath('../notebooks/results/qa4_estatisticas_20260328_163833.csv')

In [34]:
# Resumo QA4 por nível de qualidade
qa4_by_profile = all_qa4.groupby("profile").agg(
    mean_score=("score", "mean"),
    std_score=("score", "std"),
    var_score=("score", "var"),
    mean_c1=("c1_score", "mean"),
    mean_c2=("c2_score", "mean"),
    n_divergencias=("divergence_detected", "sum"),
).round(2)

print("QA4 — Estabilidade por nível:")
display(qa4_by_profile)

save_csv(qa4_by_profile.reset_index(), "qa4_resumo_por_nivel")

QA4 — Estabilidade por nível:


,mean_score,std_score,var_score,mean_c1,mean_c2,n_divergencias
profile,,,,,,
Excelente,10.00,0.00,0.00,10.00,10.00,0
Fora tema,0.00,0.00,0.00,0.00,0.00,0
Fraca,1.37,0.75,0.56,2.03,1.37,0
Intermediário,9.33,0.47,0.22,9.48,9.33,0


  💾 ..\notebooks\results\qa4_resumo_por_nivel_20260328_163833.csv


WindowsPath('../notebooks/results/qa4_resumo_por_nivel_20260328_163833.csv')

## 8. Exportar JSON + Resumo Final

In [35]:
# Salvar raw reviews completos em JSON (backup rastreável)
for run_data, name in [(cond_a, "cond_A"), (cond_b, "cond_B")]:
    path = OUTPUT_DIR / f"{name}_{TIMESTAMP}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump({
            "label": run_data["label"],
            "exam_uuid": run_data["exam_uuid"],
            "rag": run_data["rag"],
            "timestamp": TIMESTAMP,
            "raw_review": run_data["raw_review"],
        }, f, ensure_ascii=False, indent=2, default=str)
    print(f"  💾 {path}")

# QA4 JSON
qa4_path = OUTPUT_DIR / f"qa4_{TIMESTAMP}.json"
with open(qa4_path, "w", encoding="utf-8") as f:
    json.dump({
        "label": "QA4 Estabilidade",
        "num_runs": QA4_RUNS,
        "timestamp": TIMESTAMP,
        "runs": [
            {"run": i + 1, "exam_uuid": r["exam_uuid"], "raw_review": r["raw_review"]}
            for i, r in enumerate(qa4_results)
        ],
    }, f, ensure_ascii=False, indent=2, default=str)
print(f"  💾 {qa4_path}")

  💾 ..\notebooks\results\cond_A_20260328_163833.json
  💾 ..\notebooks\results\cond_B_20260328_163833.json
  💾 ..\notebooks\results\qa4_20260328_163833.json


In [36]:
# Resumo final
print("="*70)
print("  RESUMO DO EXPERIMENTO")
print("="*70)
print(f"\n  Condição A (com RAG):  média = {cond_a['df']['score'].mean():.2f}")
print(f"  Condição B (sem RAG):  média = {cond_b['df']['score'].mean():.2f}")
print(f"  Delta médio (A - B):   {comparison['Delta'].mean():+.2f}")
print(f"\n  QA4 — {QA4_RUNS} execuções (R={QA4_RUNS}, A_rep=4 níveis):")
print(f"    Variância média:           {qa4_stats['var_score'].mean():.2f}")
print(f"    Desvio padrão médio:       {qa4_stats['std_score'].mean():.2f}")
print(f"    Variação absoluta máxima:  {qa4_stats['var_abs'].max():.2f}")
qa4_ok = (qa4_stats['var_abs'] <= 1.0).all()
print(f"    QA4 aprovado (<=1.0 pt):   {'✅ SIM' if qa4_ok else '❌ NÃO'}")

print(f"\n  CSVs salvos em: {OUTPUT_DIR.resolve()}/")
print(f"    cond_A_{TIMESTAMP}.csv")
print(f"    cond_B_{TIMESTAMP}.csv")
print(f"    comparacao_A_vs_B_{TIMESTAMP}.csv")
print(f"    resumo_por_nivel_{TIMESTAMP}.csv")
print(f"    qa4_todas_runs_{TIMESTAMP}.csv")
print(f"    qa4_estatisticas_{TIMESTAMP}.csv")
print(f"    qa4_resumo_por_nivel_{TIMESTAMP}.csv")
print("="*70)

  RESUMO DO EXPERIMENTO

  Condição A (com RAG):  média = 5.09
  Condição B (sem RAG):  média = 4.86
  Delta médio (A - B):   +0.24

  QA4 — 3 execuções (R=3, A_rep=4 níveis):
    Variância média:           0.20
    Desvio padrão médio:       0.30
    Variação absoluta máxima:  1.37
    QA4 aprovado (<=1.0 pt):   ❌ NÃO

  CSVs salvos em: C:\Users\Maycon Mendes\OneDrive\Área de Trabalho\IFF\TCC\ai-grading-system\notebooks\results/
    cond_A_20260328_163833.csv
    cond_B_20260328_163833.csv
    comparacao_A_vs_B_20260328_163833.csv
    resumo_por_nivel_20260328_163833.csv
    qa4_todas_runs_20260328_163833.csv
    qa4_estatisticas_20260328_163833.csv
    qa4_resumo_por_nivel_20260328_163833.csv
